In [156]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Preprocessing

In [157]:
df = pd.read_csv("ai_student_impact_cleaned.csv")
df.head()

,Student_ID,Major_Category,Year_of_Study,Pre_Semester_GPA,Weekly_GenAI_Hours,Primary_Use_Case,Prompt_Engineering_Skill,Tool_Diversity,Paid_Subscription,Traditional_Study_Hours,Perceived_AI_Dependency,Institutional_Policy,Anxiety_Level_During_Exams,Post_Semester_GPA,Skill_Retention_Score,Burnout_Risk_Level
0,100001,Humanities,Senior,2.418,23.31,Copywriting/Drafting,Beginner,1,True,8.13,5,Allowed_With_Citation,6,2.393,86.44,High
1,100002,Medical,Junior,3.821,1.12,Ideation,Advanced,5,False,16.65,3,Allowed_With_Citation,9,3.696,69.39,Low
2,100003,Business,Freshman,3.398,21.26,Summarizing_Reading,Beginner,2,False,10.35,5,Strict_Ban,9,3.499,73.93,Medium
3,100004,Business,Senior,3.789,1.82,Copywriting/Drafting,Intermediate,4,False,15.23,2,Allowed_With_Citation,2,4.000,63.58,Medium
4,100005,STEM,Sophomore,3.635,9.29,Debugging/Troubleshooting,Advanced,4,False,12.55,4,Allowed_With_Citation,4,3.798,100.00,Medium


In [158]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 16 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Student_ID                  50000 non-null  int64  
 1   Major_Category              50000 non-null  object 
 2   Year_of_Study               50000 non-null  object 
 3   Pre_Semester_GPA            50000 non-null  float64
 4   Weekly_GenAI_Hours          50000 non-null  float64
 5   Primary_Use_Case            50000 non-null  object 
 6   Prompt_Engineering_Skill    50000 non-null  object 
 7   Tool_Diversity              50000 non-null  int64  
 8   Paid_Subscription           50000 non-null  bool   
 9   Traditional_Study_Hours     50000 non-null  float64
 10  Perceived_AI_Dependency     50000 non-null  int64  
 11  Institutional_Policy        50000 non-null  object 
 12  Anxiety_Level_During_Exams  50000 non-null  int64  
 13  Post_Semester_GPA           500

In [159]:
df.drop(columns = ["Student_ID"], axis = 1, inplace = True)

In [160]:
def major_category_encoder(data):
    if data == "STEM":
        return 0 
    elif data == "Business":
        return 1
    elif data == "Humanities":
        return 2
    elif data == "Medical":
        return 3
    elif data == "Arts":
        return 4
    
df["Major_Category"] = df["Major_Category"].apply(major_category_encoder)

In [161]:
def year_of_study_encoder(data):
    if data == "Junior":
        return 0
    elif data == "Freshman":
        return 1 
    elif data == "Senior":
        return 2
    elif data == "Sophomore":
        return 3
    elif data == "Graduate":
        return 4 

df["Year_of_Study"] = df["Year_of_Study"].apply(year_of_study_encoder)

In [162]:
def primary_use_case_encoder(data):
    if data == "Debugging/Troubleshooting":
        return 0 
    elif data == "Copywriting/Drafting":
        return 1
    elif data == "Ideation":
        return 2
    elif data == "Summarizing_Reading":
        return 3 
    elif data == "Direct_Answer_Generation":
        return 4
    
df["Primary_Use_Case"] = df["Primary_Use_Case"].apply(primary_use_case_encoder)

In [163]:
def prompt_engineering_skills_encoder(data):
    if data == "Beginner":
        return 0 
    elif data == "Intermediate":
        return 1 
    elif data == "Advanced":
        return 2

df["Prompt_Engineering_Skill"] = df["Prompt_Engineering_Skill"].apply(prompt_engineering_skills_encoder)

In [164]:
def paid_subscription_encoder(data):
    if data == False:
        return 0 
    elif data == True:
        return 1 

df["Paid_Subscription"] = df["Paid_Subscription"].apply(paid_subscription_encoder)

In [165]:
df["Institutional_Policy"].value_counts()

Institutional_Policy
Allowed_With_Citation    25224
Actively_Encouraged      14988
Strict_Ban                9788
Name: count, dtype: int64

In [166]:
def institutional_policy_encoder(data):
    if data == "Allowed_With_Citation":
        return 0 
    elif data == "Actively_Encouraged":
        return 1 
    elif data == "Strict_Ban":
        return 2

df["Institutional_Policy"] = df["Institutional_Policy"].apply(institutional_policy_encoder)

In [167]:
def burnout_risk_level_encoder(data):
    if data == "Low":
        return 0 
    elif data == "Medium":
        return 1 
    elif data == "High":
        return 2 
    
df["Burnout_Risk_Level"] = df["Burnout_Risk_Level"].apply(burnout_risk_level_encoder)

In [168]:
df.head()

,Major_Category,Year_of_Study,Pre_Semester_GPA,Weekly_GenAI_Hours,Primary_Use_Case,Prompt_Engineering_Skill,Tool_Diversity,Paid_Subscription,Traditional_Study_Hours,Perceived_AI_Dependency,Institutional_Policy,Anxiety_Level_During_Exams,Post_Semester_GPA,Skill_Retention_Score,Burnout_Risk_Level
0,2,2,2.418,23.31,1,0,1,1,8.13,5,0,6,2.393,86.44,2
1,3,0,3.821,1.12,2,2,5,0,16.65,3,0,9,3.696,69.39,0
2,1,1,3.398,21.26,3,0,2,0,10.35,5,2,9,3.499,73.93,1
3,1,2,3.789,1.82,1,1,4,0,15.23,2,0,2,4.000,63.58,1
4,0,3,3.635,9.29,0,2,4,0,12.55,4,0,4,3.798,100.00,1


In [169]:
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns = ["Post_Semester_GPA"], axis = 1), 
                                                    df["Post_Semester_GPA"], 
                                                    test_size = 0.2, 
                                                    shuffle = True, 
                                                    random_state = 42)

In [170]:
numerical_features = ["Pre_Semester_GPA", "Weekly_GenAI_Hours", "Traditional_Study_Hours", "Skill_Retention_Score"]
scaler = RobustScaler()
X_train[numerical_features] = scaler.fit_transform(X_train[numerical_features])
X_test[numerical_features] = scaler.transform(X_test[numerical_features])

In [171]:
dtr = DecisionTreeRegressor(random_state = 42)
dtr.fit(X_train, y_train)

DecisionTreeRegressor(random_state=42)

In [172]:
rfr = RandomForestRegressor(random_state = 42)
rfr.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

# Model Training

In [173]:
gbr = GradientBoostingRegressor(random_state = 42)
gbr.fit(X_train, y_train)

GradientBoostingRegressor(random_state=42)

In [174]:
etr = ExtraTreesRegressor(random_state = 42)
etr.fit(X_train, y_train)

ExtraTreesRegressor(random_state=42)

# Model Evaluation 

In [175]:
y_pred_train_dtr = dtr.predict(X_train)
mae_train_dtr = mean_absolute_error(y_train, y_pred_train_dtr)
mse_train_dtr = mean_squared_error(y_train, y_pred_train_dtr)
rmse_train_dtr = np.sqrt(mse_train_dtr)
r2_train_dtr = r2_score(y_train, y_pred_train_dtr)

y_pred_test_dtr = dtr.predict(X_test)
mae_test_dtr = mean_absolute_error(y_test, y_pred_test_dtr)
mse_test_dtr = mean_squared_error(y_test, y_pred_test_dtr)
rmse_test_dtr = np.sqrt(mse_test_dtr)
r2_test_dtr = r2_score(y_test, y_pred_test_dtr)

In [176]:
y_pred_train_rfr = rfr.predict(X_train)
mae_train_rfr = mean_absolute_error(y_train, y_pred_train_rfr)
mse_train_rfr = mean_squared_error(y_train, y_pred_train_rfr)
rmse_train_rfr = np.sqrt(mse_train_rfr)
r2_train_rfr = r2_score(y_train, y_pred_train_rfr)

y_pred_test_rfr = rfr.predict(X_test)
mae_test_rfr = mean_absolute_error(y_test, y_pred_test_rfr)
mse_test_rfr = mean_squared_error(y_test, y_pred_test_rfr)
rmse_test_rfr = np.sqrt(mse_test_rfr)
r2_test_rfr = r2_score(y_test, y_pred_test_rfr)

In [177]:
y_pred_train_gbr = gbr.predict(X_train)
mae_train_gbr = mean_absolute_error(y_train, y_pred_train_gbr)
mse_train_gbr = mean_squared_error(y_train, y_pred_train_gbr)
rmse_train_gbr = np.sqrt(mse_train_gbr)
r2_train_gbr = r2_score(y_train, y_pred_train_gbr)

y_pred_test_gbr = gbr.predict(X_test)
mae_test_gbr = mean_absolute_error(y_test, y_pred_test_gbr)
mse_test_gbr = mean_squared_error(y_test, y_pred_test_gbr)
rmse_test_gbr = np.sqrt(mse_test_gbr)
r2_test_gbr = r2_score(y_test, y_pred_test_gbr)

In [178]:
y_pred_train_etr = etr.predict(X_train)
mae_train_etr = mean_absolute_error(y_train, y_pred_train_etr)
mse_train_etr = mean_squared_error(y_train, y_pred_train_etr)
rmse_train_etr = np.sqrt(mse_train_etr)
r2_train_etr = r2_score(y_train, y_pred_train_etr)

y_pred_test_etr = etr.predict(X_test)
mae_test_etr = mean_absolute_error(y_test, y_pred_test_etr)
mse_test_etr = mean_squared_error(y_test, y_pred_test_etr)
rmse_test_etr = np.sqrt(mse_test_etr)
r2_test_etr = r2_score(y_test, y_pred_test_etr)

In [179]:
train_result_df = pd.DataFrame({"Model": ["DTR", "RFR", "GBR", "ETR"],
                                "Train MAE": [mae_train_dtr, mae_train_rfr, mae_train_gbr, mae_train_etr], 
                                "Train MSE": [mse_train_dtr, mse_train_rfr, mse_train_gbr, mse_train_etr], 
                                "Train RMSE": [rmse_train_dtr, rmse_train_rfr, rmse_train_gbr, rmse_train_etr], 
                                "Train R2": [r2_train_dtr, r2_train_rfr, r2_train_gbr, r2_train_etr]})

train_result_df

,Model,Train MAE,Train MSE,Train RMSE,Train R2
0,DTR,0.000000e+00,0.000000e+00,0.000000e+00,1.000000
1,RFR,4.523064e-02,3.410141e-03,5.839642e-02,0.986180
2,GBR,1.140527e-01,2.102652e-02,1.450052e-01,0.914788
3,ETR,3.275613e-15,1.658807e-29,4.072846e-15,1.000000


In [180]:
test_result_df = pd.DataFrame({"Model": ["DTR", "RFR", "GBR", "ETR"],
                                "Test MAE": [mae_test_dtr, mae_test_rfr, mae_test_gbr, mae_test_etr], 
                                "Test MSE": [mse_test_dtr, mse_test_rfr, mse_test_gbr, mse_test_etr], 
                                "Test RMSE": [rmse_test_dtr, rmse_test_rfr, rmse_test_gbr, rmse_test_etr], 
                                "Test R2": [r2_test_dtr, r2_test_rfr, r2_test_gbr, r2_test_etr]})

test_result_df

,Model,Test MAE,Test MSE,Test RMSE,Test R2
0,DTR,0.171608,0.049521,0.222532,0.794849
1,RFR,0.120687,0.023737,0.154069,0.901662
2,GBR,0.114357,0.020995,0.144898,0.913022
3,ETR,0.120254,0.023436,0.153088,0.902911
